# CNN-GNN HMER - Kaggle Run All

One-click Kaggle pipeline for training and evaluating the CNN-Transformer baseline and the CNN-GNN variant on CROHME. Paste your WandB API key into `WANDB_API_KEY` below, enable Internet/GPU, then run all cells.

In [ ]:
# =============================
# User options
# =============================
REPO_URL = "https://github.com/KhaiHASO/CNN-GNN-HMER.git"
WANDB_API_KEY = ""  # Paste your WandB API key here if Kaggle Secrets are not convenient.
WANDB_ENTITY = None  # Optional: set your W&B username/team, or keep None.
RUN_TARGETS = ["baseline", "cnn_gnn"]  # options: ["baseline"], ["cnn_gnn"], or both
WANDB_PROJECT = "cnn-gnn-hmer"
WANDB_GROUP = "crohme-cnn-gnn-comparison"
WANDB_TAGS = ["crohme", "hmer", "cnn-gnn", "kaggle"]
VALIDATION_YEAR = "2014"
EVAL_YEARS = ["2014", "2016", "2019"]
MAX_EPOCHS = 100
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 2
NUM_WORKERS = 5
MAX_SIZE = 320000
PRECISION = 16
CHECK_VAL_EVERY_N_EPOCH = 2
SAFE_CHECKPOINT_EVERY_N_EPOCHS = 1  # Safest: upload a recoverable checkpoint every epoch. Increase to 2/5 to save W&B storage.
KEEP_LOCAL_SAFE_CHECKPOINTS = 2
SCALE_AUG = False

# =============================
# Pipeline implementation
# =============================
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

def run(cmd, cwd=None):
    print(f"\n$ {cmd}")
    subprocess.run(["bash", "-lc", cmd], cwd=cwd, check=True)

def ensure_python_package(package):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

ensure_python_package("wandb")
ensure_python_package("pyyaml")
import yaml
import wandb

if WANDB_API_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
else:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if secret_key:
            os.environ["WANDB_API_KEY"] = secret_key
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("Missing WANDB_API_KEY. Paste it into the WANDB_API_KEY variable near the top of this notebook, or add it as a Kaggle Secret.")

wandb.login(key=os.environ["WANDB_API_KEY"])

WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "CNN-GNN-HMER"
PROJECT_ROOT = REPO_DIR / "chuyende_tamer_temp"
BASELINE_DIR = PROJECT_ROOT / "0-cnn-transformer-baseline"
CNN_GNN_DIR = PROJECT_ROOT / "1-cnn-gnn"
OUTPUT_ROOT = WORKING / "cnn_gnn_hmer_outputs"
SHARED_DATA_ROOT = WORKING / "cnn_gnn_hmer_data"
SHARED_CROHME_DIR = SHARED_DATA_ROOT / "crohme"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SHARED_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(f"git clone {REPO_URL} {REPO_DIR}")

if not BASELINE_DIR.exists() or not CNN_GNN_DIR.exists():
    raise RuntimeError(f"Expected project folders not found under {PROJECT_ROOT}")

def find_crohme_zip():
    repo_zip = PROJECT_ROOT / "data" / "CROHME.zip"
    candidates = [repo_zip] if repo_zip.exists() else []
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("CROHME.zip"))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        raise RuntimeError("CROHME.zip not found. Keep it in the repo or attach a Kaggle Dataset containing CROHME.zip.")
    candidates.sort(key=lambda p: len(str(p)))
    print(f"Using dataset zip: {candidates[0]}")
    return candidates[0]

CROHME_ZIP = find_crohme_zip()

def setup_conda():
    conda = WORKING / "miniconda" / "bin" / "conda"
    if not conda.exists():
        run("wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh")
        run("bash miniconda.sh -b -f -p /kaggle/working/miniconda")
        run("rm miniconda.sh")
    run("/kaggle/working/miniconda/bin/conda create -n tamer python=3.7 -y || true")
    run("source /kaggle/working/miniconda/bin/activate tamer && conda install pytorch-lightning=1.4.9 torchmetrics=0.6.0 pandoc=1.19.2.1 libstdcxx-ng -c conda-forge -y")

def install_project(project_dir):
    run("source /kaggle/working/miniconda/bin/activate tamer && pip install -r requirements.txt && pip install wandb pandas pyyaml && pip install -e .", cwd=project_dir)

SAFE_CALLBACK_CODE = r"""
from pathlib import Path

import pytorch_lightning as pl

try:
    import wandb
except Exception:
    wandb = None


class SafeWandbCheckpointUploader(pl.Callback):
    def __init__(self, target, run_name, every_n_epochs=1, keep_local=2):
        super().__init__()
        self.target = target
        self.run_name = run_name
        self.every_n_epochs = max(int(every_n_epochs), 1)
        self.keep_local = max(int(keep_local), 1)
        self.safe_dir = Path("safe_checkpoints") / target
        self.uploaded = set()

    def _upload(self, trainer, path, reason):
        path = Path(path)
        if not path.exists() or wandb is None or wandb.run is None:
            return
        key = f"{path.resolve()}::{path.stat().st_size}::{int(path.stat().st_mtime)}"
        if key in self.uploaded:
            return
        artifact = wandb.Artifact(
            name=f"{self.target}-{self.run_name}-safe-checkpoint",
            type="training-checkpoint",
            metadata={
                "target": self.target,
                "run_name": self.run_name,
                "reason": reason,
                "epoch": int(getattr(trainer, "current_epoch", -1)),
                "global_step": int(getattr(trainer, "global_step", -1)),
                "source_file": path.name,
            },
        )
        artifact.add_file(str(path), name=f"checkpoints/{path.name}")
        aliases = ["latest", self.target, reason, f"epoch-{int(getattr(trainer, 'current_epoch', -1))}"]
        wandb.run.log_artifact(artifact, aliases=aliases)
        wandb.run.log({f"safe_checkpoint/{self.target}_epoch": int(getattr(trainer, "current_epoch", -1))})
        self.uploaded.add(key)
        print(f"Uploaded safe checkpoint to W&B: {path} ({reason})")

    def _cleanup_local(self):
        ckpts = sorted(self.safe_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
        for old in ckpts[self.keep_local:]:
            old.unlink(missing_ok=True)

    def _save_and_upload(self, trainer, reason):
        self.safe_dir.mkdir(parents=True, exist_ok=True)
        epoch = int(getattr(trainer, "current_epoch", -1))
        step = int(getattr(trainer, "global_step", -1))
        path = self.safe_dir / f"{self.target}-epoch={epoch:04d}-step={step:08d}-{reason}.ckpt"
        trainer.save_checkpoint(str(path))
        self._upload(trainer, path, reason)
        self._cleanup_local()

    def on_train_epoch_end(self, trainer, pl_module, unused=None):
        epoch_number = int(getattr(trainer, "current_epoch", 0)) + 1
        if epoch_number % self.every_n_epochs == 0:
            self._save_and_upload(trainer, "epoch")

    def on_validation_end(self, trainer, pl_module):
        for path in Path("lightning_logs").rglob("*.ckpt"):
            self._upload(trainer, path, "validation")

    def on_exception(self, trainer, pl_module, exception):
        try:
            self._save_and_upload(trainer, "exception")
        except Exception as exc:
            print(f"Failed to save exception checkpoint: {exc}")

    def on_train_end(self, trainer, pl_module):
        self._save_and_upload(trainer, "final")
        for path in Path("lightning_logs").rglob("*.ckpt"):
            self._upload(trainer, path, "final")
"""


def write_safe_callback(project_dir):
    (project_dir / "safe_wandb_callback.py").write_text(SAFE_CALLBACK_CODE, encoding="utf-8")

def prepare_data():
    if SHARED_CROHME_DIR.exists():
        return
    extract_dir = SHARED_DATA_ROOT / "extract"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    run(f"unzip -q -o {CROHME_ZIP} -d {extract_dir}")
    candidates = [extract_dir / "CROHME_extracted" / "crohme", extract_dir / "crohme"]
    source = next((p for p in candidates if p.exists()), None)
    if source is None:
        source = next((p for p in extract_dir.rglob("dictionary.txt") if p.is_file()), None)
        if source:
            source = source.parent
    if source is None:
        raise RuntimeError(f"Could not locate extracted CROHME folder under {extract_dir}")
    shutil.copytree(source, SHARED_CROHME_DIR)

def make_config(project_dir, target):
    with open(project_dir / "config" / "crohme.yaml", "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    run_name = f"{target}-{time.strftime('%Y%m%d-%H%M%S')}"
    cfg["trainer"]["logger"] = {
        "class_path": "pytorch_lightning.loggers.WandbLogger",
        "init_args": {
            "project": WANDB_PROJECT,
            "entity": WANDB_ENTITY,
            "name": run_name,
            "group": WANDB_GROUP,
            "job_type": f"train-{target}",
            "tags": WANDB_TAGS + [target],
            "save_dir": "lightning_logs",
            "log_model": True,
        },
    }
    cfg["trainer"]["max_epochs"] = MAX_EPOCHS
    cfg["trainer"]["gpus"] = 1
    cfg["trainer"]["precision"] = PRECISION
    cfg["trainer"]["check_val_every_n_epoch"] = CHECK_VAL_EVERY_N_EPOCH
    callbacks = cfg["trainer"].setdefault("callbacks", [])
    for callback in callbacks:
        if callback.get("class_path") == "pytorch_lightning.callbacks.ModelCheckpoint":
            init_args = callback.setdefault("init_args", {})
            init_args["save_top_k"] = max(int(init_args.get("save_top_k", 1)), 3)
            init_args["save_last"] = True
            init_args["filename"] = f"{target}-{{epoch}}-{{step}}-{{val_ExpRate:.4f}}"
    callbacks.append({
        "class_path": "safe_wandb_callback.SafeWandbCheckpointUploader",
        "init_args": {
            "target": target,
            "run_name": run_name,
            "every_n_epochs": SAFE_CHECKPOINT_EVERY_N_EPOCHS,
            "keep_local": KEEP_LOCAL_SAFE_CHECKPOINTS,
        },
    })
    cfg["data"]["folder"] = str(SHARED_CROHME_DIR)
    cfg["data"]["test_folder"] = VALIDATION_YEAR
    cfg["data"]["max_size"] = MAX_SIZE
    cfg["data"]["train_batch_size"] = TRAIN_BATCH_SIZE
    cfg["data"]["eval_batch_size"] = EVAL_BATCH_SIZE
    cfg["data"]["num_workers"] = NUM_WORKERS
    cfg["data"]["scale_aug"] = SCALE_AUG
    if target == "cnn_gnn":
        cfg["model"]["use_gat"] = True
        cfg["model"].setdefault("gat_num_layers", 2)
        cfg["model"].setdefault("gat_num_heads", 8)
        cfg["model"].setdefault("gat_hidden_dim", None)
        cfg["model"].setdefault("gat_dropout", 0.1)
    else:
        for key in ["use_gat", "gat_num_layers", "gat_num_heads", "gat_hidden_dim", "gat_dropout"]:
            cfg["model"].pop(key, None)
    config_path = project_dir / "config" / f"kaggle_{target}.yaml"
    with open(config_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return config_path, run_name

EVAL_SCRIPT = r'''
import json
import shutil
import sys
from pathlib import Path
from pytorch_lightning import Trainer, seed_everything
from tamer.datamodule import HMEDatamodule
from tamer.lit_tamer import LitTAMER

seed_everything(7)
TEST_COUNTS = {'2014': 986, '2016': 1147, '2019': 1199, 'test': 24607}

ckpt = Path(sys.argv[1])
data_folder = sys.argv[2]
out_dir = Path(sys.argv[3])
years = sys.argv[4:]
out_dir.mkdir(parents=True, exist_ok=True)

for year in years:
    year_dir = out_dir / year
    year_dir.mkdir(parents=True, exist_ok=True)
    trainer = Trainer(logger=False, gpus=1)
    dm = HMEDatamodule(folder=data_folder, test_folder=year, max_size=320000, scale_to_limit=True)
    model = LitTAMER.load_from_checkpoint(str(ckpt))
    metrics = trainer.test(model, datamodule=dm)[0]
    for name in ['result.zip', 'errors.json', 'predictions.json']:
        p = Path(name)
        if p.exists():
            shutil.move(str(p), str(year_dir / name))
    errors_path = year_dir / 'errors.json'
    summary = {'raw_metrics': metrics}
    if errors_path.exists() and year in TEST_COUNTS:
        errors = json.loads(errors_path.read_text(encoding='utf-8'))
        total = TEST_COUNTS[year]
        exact = total - len(errors)
        le1 = exact + sum(1 for item in errors.values() if item['dist'] <= 1)
        le2 = exact + sum(1 for item in errors.values() if item['dist'] <= 2)
        summary.update({'ExpRate': exact / total, 'ExpRate<=1': le1 / total, 'ExpRate<=2': le2 / total})
    (year_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    with open(year_dir / 'summary.txt', 'w', encoding='utf-8') as f:
        for key, value in summary.items():
            f.write(f'{key}: {value}\n')
'''

def write_eval_script(project_dir):
    script_path = project_dir / "kaggle_eval.py"
    script_path.write_text(EVAL_SCRIPT, encoding="utf-8")
    return script_path

def latest_checkpoint(project_dir):
    ckpts = list((project_dir / "lightning_logs").rglob("*.ckpt"))
    if not ckpts:
        raise RuntimeError(f"No checkpoint found in {project_dir / 'lightning_logs'}")
    ckpts.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return ckpts[0]

def upload_artifact(target, run_name, output_dir, checkpoint_path, config_path):
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=f"{run_name}-eval-upload", group=WANDB_GROUP, job_type=f"eval-upload-{target}", tags=WANDB_TAGS + [target, "eval", "checkpoint"], reinit=True)
    artifact = wandb.Artifact(f"{target}-{run_name}-checkpoint-eval", type="model-eval", metadata={"target": target, "validation_year": VALIDATION_YEAR, "eval_years": EVAL_YEARS})
    artifact.add_file(str(checkpoint_path), name=f"checkpoint/{checkpoint_path.name}")
    artifact.add_file(str(config_path), name=f"config/{config_path.name}")
    artifact.add_dir(str(output_dir), name="eval")
    run.log_artifact(artifact)
    wandb.finish()

TARGETS = {
    "baseline": BASELINE_DIR,
    "cnn_gnn": CNN_GNN_DIR,
}

setup_conda()

for target in RUN_TARGETS:
    if target not in TARGETS:
        raise ValueError(f"Unknown target: {target}")
    project_dir = TARGETS[target]
    print(f"\n==================== {target.upper()} ====================")
    install_project(project_dir)
    prepare_data()
    config_path, run_name = make_config(project_dir, target)
    run(f"source /kaggle/working/miniconda/bin/activate tamer && python train.py --config {config_path.relative_to(project_dir)}", cwd=project_dir)
    ckpt = latest_checkpoint(project_dir)
    print(f"Best/latest checkpoint: {ckpt}")
    eval_script = write_eval_script(project_dir)
    output_dir = OUTPUT_ROOT / target
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    years = " ".join(EVAL_YEARS)
    run(f"source /kaggle/working/miniconda/bin/activate tamer && python {eval_script.name} {ckpt} {SHARED_CROHME_DIR} {output_dir} {years}", cwd=project_dir)
    upload_artifact(target, run_name, output_dir, ckpt, config_path)

print(f"\nDone. Local outputs are in: {OUTPUT_ROOT}")
